In [1]:
!pip install GEOparse -q

import GEOparse
import pandas as pd
import os

os.makedirs('data/raw/GSE98320', exist_ok=True)

In [3]:
from google.colab import drive
drive.mount('/content/drive')

# point raw data dir to Drive instead of local Colab storage
DATA_DIR = '/content/drive/MyDrive/MIRAGE/data/raw/GSE98320'
os.makedirs(DATA_DIR, exist_ok=True)

Mounted at /content/drive


In [4]:
soft_file_check = os.path.join(DATA_DIR, "GSE98320_family.soft.gz")

if not os.path.exists(soft_file_check):
    gse = GEOparse.get_GEO(geo="GSE98320", destdir=DATA_DIR)
else:
    gse = GEOparse.get_GEO(filepath=soft_file_check)

print(f"Number of samples (GSMs): {len(gse.gsms)}")

06-Sep-2026 02:53:08 DEBUG utils - Directory /content/drive/MyDrive/MIRAGE/data/raw/GSE98320 already exists. Skipping.
DEBUG:GEOparse:Directory /content/drive/MyDrive/MIRAGE/data/raw/GSE98320 already exists. Skipping.
06-Sep-2026 02:53:08 INFO GEOparse - Downloading ftp://ftp.ncbi.nlm.nih.gov/geo/series/GSE98nnn/GSE98320/soft/GSE98320_family.soft.gz to /content/drive/MyDrive/MIRAGE/data/raw/GSE98320/GSE98320_family.soft.gz
INFO:GEOparse:Downloading ftp://ftp.ncbi.nlm.nih.gov/geo/series/GSE98nnn/GSE98320/soft/GSE98320_family.soft.gz to /content/drive/MyDrive/MIRAGE/data/raw/GSE98320/GSE98320_family.soft.gz
100%|██████████| 537M/537M [00:07<00:00, 70.4MB/s]
06-Sep-2026 02:53:17 DEBUG downloader - Size validation passed
DEBUG:GEOparse:Size validation passed
06-Sep-2026 02:53:17 DEBUG downloader - Moving /tmp/tmpf18oo8of to /content/drive/MyDrive/MIRAGE/data/raw/GSE98320/GSE98320_family.soft.gz
DEBUG:GEOparse:Moving /tmp/tmpf18oo8of to /content/drive/MyDrive/MIRAGE/data/raw/GSE98320/GSE983

Number of samples (GSMs): 1208


In [5]:
sample_id = list(gse.gsms.keys())[0]
sample_gsm = gse.gsms[sample_id]

print(f"GSM ID: {sample_id}\n")
for key, value in sample_gsm.metadata.items():
    print(f"{key}: {value}")

GSM ID: GSM2590943

title: ['kidney biopsy, archetype1']
geo_accession: ['GSM2590943']
status: ['Public on Jul 10 2017']
submission_date: ['Apr 28 2017']
last_update_date: ['Jun 07 2022']
type: ['RNA']
channel_count: ['1']
source_name_ch1: ['kidney biopsy']
organism_ch1: ['Homo sapiens']
taxid_ch1: ['9606']
characteristics_ch1: ['tissue: kidney', 'archetype cluster: 2', 'd96: TCMR', 'g: 0', 'cg: 0', 'i: 2', 'ci: 3', 't: 3', 'ct: 3', 'v: 0', 'cv: 1', 'ah: NA', 'ptc: 0', 'c4d: 0', 'in i-ifta set n=234?: 0', 'mmdx: -', 'ifta (0, <10%, ≥10%): -']
treatment_protocol_ch1: ['The immunosuppressive treatment of the patients before the biopsy was based on individual treatment regiments; the treatment after the biopsy was adjusted based on the histopathological diagnosis.']
growth_protocol_ch1: ['All human kidney biopsies taken for clinical indication during the period specified above were included. Tissue was immediately placed in RNAlater.']
molecule_ch1: ['total RNA']
extract_protocol_ch1: ['T

In [6]:
def parse_characteristics(gsm):
    """Turn a GSM's characteristics_ch1 list of 'key: value' strings into a dict."""
    parsed = {'GSM': gsm.metadata['geo_accession'][0]}
    for entry in gsm.metadata.get('characteristics_ch1', []):
        if ': ' in entry:
            key, value = entry.split(': ', 1)
            parsed[key.strip()] = value.strip()
    return parsed

records = [parse_characteristics(gsm) for gsm in gse.gsms.values()]
meta_df = pd.DataFrame(records)

print(meta_df.shape)
meta_df.head()

(1208, 18)


,GSM,tissue,archetype cluster,d96,g,cg,i,ci,t,ct,v,cv,ah,ptc,c4d,in i-ifta set n=234?,mmdx,"ifta (0, <10%, ≥10%)"
0,GSM2590943,kidney,2,TCMR,0,0,2,3,3,3,0,1,NA,0,0,0,-,-
1,GSM2590944,kidney,4,TCMR,0,1,3,2,3,2,0,1,1,1,1,0,-,-
2,GSM2590945,kidney,1,Bord.,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,0,1,NR,<10%
3,GSM2590946,kidney,5,ABMR,1,1,0,2,0,2,0,2,3,1,0,0,-,-
4,GSM2590947,kidney,1,AKI,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,0,1,NR,0


In [7]:
print("=== d96 value counts (raw diagnostic labels) ===")
print(meta_df['d96'].value_counts(dropna=False))

print("\n=== mmdx value counts ===")
print(meta_df['mmdx'].value_counts(dropna=False))

print("\n=== Missingness per column (count of 'NA' string) ===")
print((meta_df == 'NA').sum().sort_values(ascending=False))

=== d96 value counts (raw diagnostic labels) ===
d96
NOMOA       274
ABMR        215
IFTA        145
Bord.       109
GN           97
AKI          96
TCMR         87
Mixed        41
TG           40
BK           37
Other        25
ABMRsusp     24
DiabNeph     18
Name: count, dtype: int64

=== mmdx value counts ===
mmdx
-        974
NR       132
ABMR      66
TCMR      24
Mixed     12
Name: count, dtype: int64

=== Missingness per column (count of 'NA' string) ===
cv                      345
v                       304
ah                      290
ptc                     271
ci                      269
cg                      268
ct                      268
g                       263
t                       254
i                       253
tissue                    0
GSM                       0
d96                       0
archetype cluster         0
c4d                       0
in i-ifta set n=234?      0
mmdx                      0
ifta (0, <10%, ≥10%)      0
dtype: int64


In [8]:
print("=== Archetype cluster value counts ===")
print(meta_df['archetype cluster'].value_counts().sort_index())

print("\n=== Cross-tab: archetype cluster vs d96 ===")
crosstab_d96 = pd.crosstab(meta_df['archetype cluster'], meta_df['d96'])
print(crosstab_d96)

print("\n=== Cross-tab: archetype cluster vs mmdx ===")
crosstab_mmdx = pd.crosstab(meta_df['archetype cluster'], meta_df['mmdx'])
print(crosstab_mmdx)

=== Archetype cluster value counts ===
archetype cluster
1    774
2     81
3     27
4    139
5    136
6     51
Name: count, dtype: int64

=== Cross-tab: archetype cluster vs d96 ===
d96                ABMR  ABMRsusp  AKI  BK  Bord.  DiabNeph  GN  IFTA  Mixed  \
archetype cluster                                                              
1                    54        11   90  20     79        15  77   123      3   
2                     2         1    0  15      7         0   5     2      6   
3                     3         1    0   0      1         0   0     1      9   
4                    48         3    6   1     17         0   8    12      5   
5                    84         5    0   1      3         0   5     3     17   
6                    24         3    0   0      2         3   2     4      1   

d96                NOMOA  Other  TCMR  TG  
archetype cluster                          
1                    240     19    27  16  
2                      3      2    37   1  
3

In [9]:
archetype_to_label = {
    1: 'NR',
    2: 'TCMR',
    3: 'Mixed',
    4: 'ABMR',  # early ABMR
    5: 'ABMR',  # fully developed ABMR
    6: 'ABMR',  # late ABMR
}

meta_df['archetype cluster'] = meta_df['archetype cluster'].astype(int)
meta_df['mirage_label'] = meta_df['archetype cluster'].map(archetype_to_label)

print(meta_df['mirage_label'].value_counts())
print("\nTotal:", meta_df['mirage_label'].value_counts().sum())
print("Any unmapped (NaN)?", meta_df['mirage_label'].isna().sum())

mirage_label
NR       774
ABMR     326
TCMR      81
Mixed     27
Name: count, dtype: int64

Total: 1208
Any unmapped (NaN)? 0


In [10]:
import numpy as np

banff_ordinal_cols = ['g', 'cg', 'i', 'ci', 't', 'ct', 'v', 'cv', 'ah', 'ptc']

# replace the string 'NA' with real np.nan, but ONLY in pathology-relevant columns
# (leave other columns untouched for now, in case 'NA' means something different elsewhere)
for col in banff_ordinal_cols + ['c4d', 'in i-ifta set n=234?', 'ifta (0, <10%, ≥10%)']:
    meta_df[col] = meta_df[col].replace('NA', np.nan)

# check what unique raw values exist in each column before converting to numeric
for col in banff_ordinal_cols + ['c4d', 'in i-ifta set n=234?', 'ifta (0, <10%, ≥10%)']:
    print(f"{col}: {sorted(meta_df[col].dropna().unique(), key=str)}")
    print()

g: ['#N/A!', '0', '1', '2', '3']

cg: ['#N/A!', '0', '1', '2', '3']

i: ['#N/A!', '0', '1', '2', '3']

ci: ['#N/A!', '0', '1', '2', '3']

t: ['#N/A!', '0', '1', '2', '3']

ct: ['#N/A!', '0', '1', '2', '3']

v: ['#N/A!', '0', '1', '2', '3']

cv: ['#N/A!', '0', '1', '2', '3']

ah: ['#N/A!', '0', '1', '2', '3']

ptc: ['#N/A!', '0', '1', '2', '3']

c4d: ['#N/A!', '0', '1', '2', '3', '4', 'Neg', 'Not done', 'Pending', 'Pos', 'focal mild', 'neg', 'pos', 'pos trace', 'positive', 'trace']

in i-ifta set n=234?: ['0', '1']

ifta (0, <10%, ≥10%): ['-', '0', '<10%', '≥10%']



In [11]:
# --- Step 1: unify missing markers across all pathology columns ---
missing_markers = ['NA', '#N/A!']

banff_ordinal_cols = ['g', 'cg', 'i', 'ci', 't', 'ct', 'v', 'cv', 'ah', 'ptc']

for col in banff_ordinal_cols + ['c4d', 'in i-ifta set n=234?']:
    meta_df[col] = meta_df[col].replace(missing_markers, np.nan)

meta_df['ifta (0, <10%, ≥10%)'] = meta_df['ifta (0, <10%, ≥10%)'].replace(['-'] + missing_markers, np.nan)

# --- Step 2: cast Banff ordinal scores (0-3) to numeric ---
for col in banff_ordinal_cols:
    meta_df[col] = pd.to_numeric(meta_df[col], errors='coerce')

# --- Step 3: encode c4d as 3-level ordinal (negative=0, trace/focal=1, clear-positive=2) ---
c4d_map = {
    '0': 0, 'Neg': 0, 'neg': 0,
    '1': 1, 'trace': 1, 'pos trace': 1, 'focal mild': 1,
    '2': 2, '3': 2, '4': 2, 'Pos': 2, 'pos': 2, 'positive': 2,
    'Not done': np.nan, 'Pending': np.nan,
}
meta_df['c4d_ordinal'] = meta_df['c4d'].map(c4d_map)

# sanity check: any values NOT captured by the map?
unmapped = meta_df.loc[meta_df['c4d'].notna() & meta_df['c4d_ordinal'].isna(), 'c4d'].unique()
print("Unmapped c4d values (should be empty):", unmapped)

# --- Step 4: encode ifta as 3-level ordinal ---
ifta_map = {'0': 0, '<10%': 1, '≥10%': 2}
meta_df['ifta_ordinal'] = meta_df['ifta (0, <10%, ≥10%)'].map(ifta_map)

# --- Step 5: quick missingness summary on final pathology features ---
pathology_feature_cols = banff_ordinal_cols + ['c4d_ordinal', 'ifta_ordinal', 'in i-ifta set n=234?']
print("\n=== Missingness (%) per pathology feature ===")
print((meta_df[pathology_feature_cols].isna().mean() * 100).round(1))

Unmapped c4d values (should be empty): ['Not done' 'Pending']

=== Missingness (%) per pathology feature ===
g                       21.9
cg                      22.3
i                       21.0
ci                      22.4
t                       21.1
ct                      22.3
v                       25.2
cv                      28.6
ah                      24.1
ptc                     22.5
c4d_ordinal              2.2
ifta_ordinal            80.6
in i-ifta set n=234?     0.0
dtype: float64


In [12]:
# Feature columns for the model (pathology modality)
pathology_features = banff_ordinal_cols + ['c4d_ordinal']
# ifta_ordinal deliberately excluded from primary feature set — only valid for n=234 subset

# Fields that must NEVER enter the feature matrix (they define or leak the target)
label_deriving_cols = ['archetype cluster', 'd96', 'mmdx']

# Build the clean biopsy-level table
clean_df = meta_df[['GSM', 'mirage_label'] + pathology_features +
                    ['ifta_ordinal', 'in i-ifta set n=234?'] +  # kept for secondary/subset analysis, not primary features
                    label_deriving_cols].copy()  # kept for reference/audit trail, will be dropped before modeling

print(clean_df.shape)
clean_df.head()

(1208, 18)


,GSM,mirage_label,g,cg,i,ci,t,ct,v,cv,ah,ptc,c4d_ordinal,ifta_ordinal,in i-ifta set n=234?,archetype cluster,d96,mmdx
0,GSM2590943,TCMR,0.0,0.0,2.0,3.0,3.0,3.0,0.0,1.0,NaN,0.0,0.0,NaN,0,2,TCMR,-
1,GSM2590944,ABMR,0.0,1.0,3.0,2.0,3.0,2.0,0.0,1.0,1.0,1.0,1.0,NaN,0,4,TCMR,-
2,GSM2590945,NR,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,1.0,1,1,Bord.,NR
3,GSM2590946,ABMR,1.0,1.0,0.0,2.0,0.0,2.0,0.0,2.0,3.0,1.0,0.0,NaN,0,5,ABMR,-
4,GSM2590947,NR,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,1,1,AKI,NR


In [13]:
output_path = os.path.join(DATA_DIR, 'gse98320_biopsy_metadata_clean.csv')
clean_df.to_csv(output_path, index=False)
print(f"Saved to {output_path}")

Saved to /content/drive/MyDrive/MIRAGE/data/raw/GSE98320/gse98320_biopsy_metadata_clean.csv


In [14]:
# inspect one sample's expression table structure first, since column names vary by platform
sample_id = list(gse.gsms.keys())[0]
print(gse.gsms[sample_id].table.head())
print("\nColumns:", gse.gsms[sample_id].table.columns.tolist())
print("Number of probes:", len(gse.gsms[sample_id].table))

          ID_REF     VALUE
0    11715100_at  4.095828
1  11715101_s_at  6.464149
2  11715102_x_at  4.366191
3  11715103_x_at  6.135984
4  11715104_s_at  6.109552

Columns: ['ID_REF', 'VALUE']
Number of probes: 49395


In [15]:
expression_dict = {}

for gsm_id, gsm in gse.gsms.items():
    series = gsm.table.set_index('ID_REF')['VALUE']
    expression_dict[gsm_id] = series

expr_matrix = pd.DataFrame(expression_dict)

print("Shape (probes x samples):", expr_matrix.shape)
expr_matrix.iloc[:5, :5]

Shape (probes x samples): (49395, 1208)


,GSM2590943,GSM2590944,GSM2590945,GSM2590946,GSM2590947
ID_REF,,,,,
11715100_at,4.095828,4.736311,4.046332,4.376275,4.794849
11715101_s_at,6.464149,6.644839,6.081870,6.101919,6.298720
11715102_x_at,4.366191,4.538867,4.102110,4.432054,4.647658
11715103_x_at,6.135984,5.620510,5.003925,5.657558,4.732664
11715104_s_at,6.109552,6.192753,6.031128,6.123824,6.140103


In [16]:
# confirm no probe rows got silently dropped/misaligned across samples
print("Any all-NaN rows?", expr_matrix.isna().all(axis=1).sum())
print("Any all-NaN columns?", expr_matrix.isna().all(axis=0).sum())

# confirm every GSM in our pathology table also has expression data, and vice versa
gsms_in_expr = set(expr_matrix.columns)
gsms_in_meta = set(clean_df['GSM'])
print("In pathology but not expression:", len(gsms_in_meta - gsms_in_expr))
print("In expression but not pathology:", len(gsms_in_expr - gsms_in_meta))

Any all-NaN rows? 0
Any all-NaN columns? 0
In pathology but not expression: 0
In expression but not pathology: 0


In [17]:
expr_output_path = os.path.join(DATA_DIR, 'gse98320_expression_raw_probes.parquet')
expr_matrix.to_parquet(expr_output_path)
print(f"Saved to {expr_output_path}")
print(f"File size: {os.path.getsize(expr_output_path) / 1e6:.1f} MB")

Saved to /content/drive/MyDrive/MIRAGE/data/raw/GSE98320/gse98320_expression_raw_probes.parquet
File size: 583.3 MB


In [18]:
# GEOparse can fetch platform annotation directly
gpl = GEOparse.get_GEO(geo="GPL15207", destdir=DATA_DIR)

print(gpl.table.head())
print("\nColumns:", gpl.table.columns.tolist())
print("Number of probes in annotation:", len(gpl.table))

06-Sep-2026 03:12:16 DEBUG utils - Directory /content/drive/MyDrive/MIRAGE/data/raw/GSE98320 already exists. Skipping.
DEBUG:GEOparse:Directory /content/drive/MyDrive/MIRAGE/data/raw/GSE98320 already exists. Skipping.
06-Sep-2026 03:12:16 INFO GEOparse - Downloading http://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?targ=self&acc=GPL15207&form=text&view=full to /content/drive/MyDrive/MIRAGE/data/raw/GSE98320/GPL15207.txt
INFO:GEOparse:Downloading http://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?targ=self&acc=GPL15207&form=text&view=full to /content/drive/MyDrive/MIRAGE/data/raw/GSE98320/GPL15207.txt
06-Sep-2026 03:12:17 DEBUG downloader - Total size: 0
DEBUG:GEOparse:Total size: 0
06-Sep-2026 03:12:17 DEBUG downloader - md5: None
DEBUG:GEOparse:md5: None
297MB [00:08, 36.6MB/s]
06-Sep-2026 03:12:25 DEBUG downloader - Moving /tmp/tmpx321hvl0 to /content/drive/MyDrive/MIRAGE/data/raw/GSE98320/GPL15207.txt
DEBUG:GEOparse:Moving /tmp/tmpx321hvl0 to /content/drive/MyDrive/MIRAGE/data/raw/GSE98320/G

              ID                GeneChip Array Species Scientific Name  \
0    11715100_at  Human Genome PrimeView Array            Homo sapiens   
1  11715101_s_at  Human Genome PrimeView Array            Homo sapiens   
2  11715102_x_at  Human Genome PrimeView Array            Homo sapiens   
3  11715103_x_at  Human Genome PrimeView Array            Homo sapiens   
4  11715104_s_at  Human Genome PrimeView Array            Homo sapiens   

  Annotation Date       Sequence Type                  Sequence Source  \
0       30-Mar-16  Consensus sequence  Affymetrix Proprietary Database   
1       30-Mar-16  Consensus sequence  Affymetrix Proprietary Database   
2       30-Mar-16  Consensus sequence  Affymetrix Proprietary Database   
3       30-Mar-16  Consensus sequence  Affymetrix Proprietary Database   
4       30-Mar-16  Consensus sequence  Affymetrix Proprietary Database   

  Transcript ID(Array Design)  \
0                   g21264570   
1                   g21264570   
2          

In [19]:
annot = gpl.table[['ID', 'Gene Symbol']].copy()

# check for multi-symbol entries and missing values
print("Sample of Gene Symbol values:")
print(annot['Gene Symbol'].dropna().sample(10, random_state=42).tolist())

print("\nMissing/empty Gene Symbol count:", annot['Gene Symbol'].isna().sum() + (annot['Gene Symbol'] == '').sum())

print("\nProbes with multiple symbols (containing '///'):", annot['Gene Symbol'].astype(str).str.contains('///').sum())

# check for AFFX control probes specifically
affx_probes = annot['ID'].astype(str).str.startswith('AFFX')
print("\nAFFX control probes:", affx_probes.sum())

Sample of Gene Symbol values:
['KIF1B', 'RBM19', 'ECE2', 'STARD13', 'CDH26', 'ARFGAP1', 'ME1', 'CSNK1G3', 'SLC13A3', 'SHMT1']

Missing/empty Gene Symbol count: 23

Probes with multiple symbols (containing '///'): 1994

AFFX control probes: 102


In [20]:
# Step 1: clean the annotation table
annot_clean = gpl.table[['ID', 'Gene Symbol']].copy()
annot_clean['Gene Symbol'] = annot_clean['Gene Symbol'].astype(str).str.strip()

# drop AFFX control probes
annot_clean = annot_clean[~annot_clean['ID'].astype(str).str.startswith('AFFX')]

# drop missing/empty symbols
annot_clean = annot_clean[annot_clean['Gene Symbol'].notna() & (annot_clean['Gene Symbol'] != '') & (annot_clean['Gene Symbol'] != 'nan')]

# drop multi-symbol (ambiguous) probes
annot_clean = annot_clean[~annot_clean['Gene Symbol'].str.contains('///')]

print(f"Probes remaining after filtering: {len(annot_clean)} (of 49,395)")
print(f"Unique genes: {annot_clean['Gene Symbol'].nunique()}")

# Step 2: map probes to genes and collapse via mean
probe_to_gene = annot_clean.set_index('ID')['Gene Symbol']

# only keep expression rows for probes that survived filtering
expr_filtered = expr_matrix.loc[expr_matrix.index.intersection(probe_to_gene.index)]

# attach gene symbol, then group by gene and average
expr_filtered = expr_filtered.copy()
expr_filtered['gene'] = probe_to_gene.loc[expr_filtered.index]

gene_expr_matrix = expr_filtered.groupby('gene').mean()

print(f"\nFinal gene-level matrix shape: {gene_expr_matrix.shape}")
gene_expr_matrix.iloc[:5, :5]

Probes remaining after filtering: 47299 (of 49,395)
Unique genes: 18836

Final gene-level matrix shape: (18836, 1208)


,GSM2590943,GSM2590944,GSM2590945,GSM2590946,GSM2590947
gene,,,,,
---,4.425787,4.275938,4.391880,4.390565,4.391024
A1BG,5.625817,4.769765,4.658133,4.618333,4.482966
A1CF,5.874190,7.821020,8.562220,8.736789,8.667558
A2M,11.519476,11.465898,11.099582,11.754741,11.078614
A2ML1,3.442411,3.105715,3.403093,3.082437,3.348595


In [21]:
# check how many probes/genes got caught under this placeholder
print("Value counts for suspicious placeholder symbols:")
print(annot_clean['Gene Symbol'].value_counts().head(10))

Value counts for suspicious placeholder symbols:
Gene Symbol
---         430
NF1          21
FMNL1        17
DMKN         16
NFATC4       16
YME1L1       15
ABI1         15
BCL2L11      14
HLA-DPB1     13
SLC12A4      13
Name: count, dtype: int64


In [22]:
# expand the exclusion list for non-gene placeholder symbols
invalid_symbols = ['nan', '', '---', '--', '-']

annot_clean = gpl.table[['ID', 'Gene Symbol']].copy()
annot_clean['Gene Symbol'] = annot_clean['Gene Symbol'].astype(str).str.strip()

annot_clean = annot_clean[~annot_clean['ID'].astype(str).str.startswith('AFFX')]
annot_clean = annot_clean[~annot_clean['Gene Symbol'].isin(invalid_symbols)]
annot_clean = annot_clean[~annot_clean['Gene Symbol'].str.contains('///')]

print(f"Probes remaining after filtering: {len(annot_clean)} (of 49,395)")
print(f"Unique genes: {annot_clean['Gene Symbol'].nunique()}")

probe_to_gene = annot_clean.set_index('ID')['Gene Symbol']
expr_filtered = expr_matrix.loc[expr_matrix.index.intersection(probe_to_gene.index)].copy()
expr_filtered['gene'] = probe_to_gene.loc[expr_filtered.index]

gene_expr_matrix = expr_filtered.groupby('gene').mean()

print(f"\nFinal gene-level matrix shape: {gene_expr_matrix.shape}")

# confirm the placeholder is gone
print("'---' still present as gene?", '---' in gene_expr_matrix.index)

Probes remaining after filtering: 46869 (of 49,395)
Unique genes: 18835

Final gene-level matrix shape: (18835, 1208)
'---' still present as gene? False


In [23]:
gene_expr_output_path = os.path.join(DATA_DIR, 'gse98320_gene_expression.parquet')
gene_expr_matrix.to_parquet(gene_expr_output_path)
print(f"Saved to {gene_expr_output_path}")
print(f"File size: {os.path.getsize(gene_expr_output_path) / 1e6:.1f} MB")

# final check: confirm gene_expr_matrix columns (GSMs) still exactly match clean_df GSMs
gsms_in_gene_expr = set(gene_expr_matrix.columns)
gsms_in_clean = set(clean_df['GSM'])
print("\nMismatch check:")
print("In pathology but not gene expression:", len(gsms_in_clean - gsms_in_gene_expr))
print("In gene expression but not pathology:", len(gsms_in_gene_expr - gsms_in_clean))

Saved to /content/drive/MyDrive/MIRAGE/data/raw/GSE98320/gse98320_gene_expression.parquet
File size: 224.5 MB

Mismatch check:
In pathology but not gene expression: 0
In gene expression but not pathology: 0


In [24]:
# --- Duplicate sample check: are any two GSMs suspiciously identical? ---
# (correlation-based check on a subsample of genes for speed)
sample_subset = gene_expr_matrix.sample(n=2000, random_state=42)  # subsample genes for speed
corr_matrix = sample_subset.corr()

# find pairs with near-perfect correlation (excluding self-correlation)
import numpy as np
corr_vals = corr_matrix.values.copy()
np.fill_diagonal(corr_vals, 0)
max_corr_per_sample = corr_vals.max(axis=0)

suspiciously_high = (max_corr_per_sample > 0.995).sum()
print(f"Samples with >0.995 correlation to another sample: {suspiciously_high}")

# --- Outlier detection: samples with unusual overall expression distributions ---
sample_means = gene_expr_matrix.mean(axis=0)
sample_stds = gene_expr_matrix.std(axis=0)

print(f"\nSample mean expression - min: {sample_means.min():.2f}, max: {sample_means.max():.2f}, median: {sample_means.median():.2f}")
print(f"Sample std expression - min: {sample_stds.min():.2f}, max: {sample_stds.max():.2f}, median: {sample_stds.median():.2f}")

# flag samples whose mean is >3 SD from the cross-sample mean
z_scores = (sample_means - sample_means.mean()) / sample_means.std()
outlier_samples = z_scores[abs(z_scores) > 3]
print(f"\nSamples with outlier mean expression (|z|>3): {len(outlier_samples)}")
if len(outlier_samples) > 0:
    print(outlier_samples)

Samples with >0.995 correlation to another sample: 6

Sample mean expression - min: 6.32, max: 6.80, median: 6.37
Sample std expression - min: 1.61, max: 2.04, median: 1.95

Samples with outlier mean expression (|z|>3): 11
GSM2591105    14.158688
GSM2591202     5.887496
GSM2591280    12.279208
GSM2591379     3.086042
GSM2591482     3.760549
GSM2591610    14.170076
GSM2591915     3.208677
GSM2591953     3.092928
GSM2592001     4.301670
GSM2592052     3.148047
GSM2592127     4.092100
dtype: float64


In [25]:
# --- Investigate the duplicate/high-correlation pairs ---
# find which specific GSM pairs are highly correlated
high_corr_pairs = []
corr_matrix_full = sample_subset.corr()  # reuse the correlation matrix from before
cols = corr_matrix_full.columns

for i in range(len(cols)):
    for j in range(i+1, len(cols)):
        if corr_matrix_full.iloc[i, j] > 0.995:
            high_corr_pairs.append((cols[i], cols[j], corr_matrix_full.iloc[i, j]))

print("High-correlation pairs (possible duplicates/replicates):")
for pair in high_corr_pairs:
    print(pair)

# --- Investigate the outlier samples: are they corrupted or genuinely extreme? ---
outlier_ids = ['GSM2591105', 'GSM2591202', 'GSM2591280', 'GSM2591379', 'GSM2591482',
               'GSM2591610', 'GSM2591915', 'GSM2591953', 'GSM2592001', 'GSM2592052', 'GSM2592127']

print("\n=== Outlier sample expression distributions ===")
print(gene_expr_matrix[outlier_ids].describe())

High-correlation pairs (possible duplicates/replicates):
('GSM2591017', 'GSM2591813', np.float64(0.9958392089535042))
('GSM2591474', 'GSM2591929', np.float64(0.9977123675643941))
('GSM2591726', 'GSM2592043', np.float64(0.9971849453285002))

=== Outlier sample expression distributions ===
         GSM2591105    GSM2591202    GSM2591280    GSM2591379    GSM2591482  \
count  18835.000000  18835.000000  18835.000000  18835.000000  18835.000000   
mean       6.803428      6.550113      6.745867      6.464315      6.484972   
std        1.722514      1.610783      1.768137      1.834871      1.771168   
min        2.649799      2.511435      2.534053      2.402339      2.562793   
25%        5.530090      5.485981      5.442731      5.136598      5.227262   
50%        6.787312      6.448943      6.694224      6.396876      6.417042   
75%        7.936710      7.445833      7.898022      7.658879      7.561968   
max       14.513875     14.597774     14.433500     14.601822     14.220890   


In [26]:
duplicate_pairs = [('GSM2591017', 'GSM2591813'), ('GSM2591474', 'GSM2591929'), ('GSM2591726', 'GSM2592043')]

for gsm_a, gsm_b in duplicate_pairs:
    title_a = gse.gsms[gsm_a].metadata['title'][0]
    title_b = gse.gsms[gsm_b].metadata['title'][0]
    label_a = clean_df.loc[clean_df['GSM'] == gsm_a, 'mirage_label'].values[0]
    label_b = clean_df.loc[clean_df['GSM'] == gsm_b, 'mirage_label'].values[0]
    print(f"{gsm_a} ({title_a}, label={label_a})  <->  {gsm_b} ({title_b}, label={label_b})")

GSM2591017 (kidney biopsy, archetype1065, label=ABMR)  <->  GSM2591813 (kidney biopsy, archetype69, label=ABMR)
GSM2591474 (kidney biopsy, archetype39, label=ABMR)  <->  GSM2591929 (kidney biopsy, archetype794, label=NR)
GSM2591726 (kidney biopsy, archetype610, label=ABMR)  <->  GSM2592043 (kidney biopsy, archetype897, label=ABMR)


In [27]:
# Assign a group ID: samples in a highly-correlated pair share a group; everyone else is their own group
clean_df['split_group'] = clean_df['GSM']  # default: each sample is its own group

duplicate_pairs = [('GSM2591017', 'GSM2591813'), ('GSM2591474', 'GSM2591929'), ('GSM2591726', 'GSM2592043')]

for gsm_a, gsm_b in duplicate_pairs:
    group_id = f"grp_{gsm_a}"
    clean_df.loc[clean_df['GSM'].isin([gsm_a, gsm_b]), 'split_group'] = group_id

print("Number of unique split groups:", clean_df['split_group'].nunique())
print("(should be 1208 - 3 = 1205, since 3 pairs collapse into 3 groups of 2)")

# Save the finalized master pathology/label table
master_path = os.path.join(DATA_DIR, 'gse98320_master_pathology_labels.csv')
clean_df.to_csv(master_path, index=False)
print(f"\nSaved master table to {master_path}")
print(f"Final shape: {clean_df.shape}")

Number of unique split groups: 1205
(should be 1208 - 3 = 1205, since 3 pairs collapse into 3 groups of 2)

Saved master table to /content/drive/MyDrive/MIRAGE/data/raw/GSE98320/gse98320_master_pathology_labels.csv
Final shape: (1208, 19)
